In [1]:
import numpy as np
import dimod
import matplotlib.pyplot as plt
from itertools import product
import yfinance as yf

**CELL 2**

Portfolio weights are continuous s.t w ∈ {0,1} so to convert to binary for QUBO we represent each continuous weight as a sum of binary digits. So given asset i, its weight is encoded as wi = (1/2^n_bits - 1)*sum(2^k*xi,k).
So for n_bits=3 that I chose, each asset gets 3 binary variables (xi,0 , xi,1 , xi,2) and the possible values of wi are onw of 8 discerte levels {0,1/7,2/7,...,7/7}.

**Return term:**
Expected portfolio return is sum(mu*weights). Substitute the binary encoding of wi to get sum(mu)*((1/2^n_bits - 1)*sum(2^k*xi,k)) which is linear in binary variables, so it only touches the diagonal of Q. By **binary idempotency** (where x^2=x for binary x) each coefficient mu*(2^k/2^n_bits -1) will land on Q[idx,idx].

**Risk term:**
Portfolio varaince is w^TCw = ∑i∑j∑ijwiwj. Substitute the binary encoding to get λ∑i∑j∑k∑l∑ij*(2^k/2^n_bits-1)(2^l/2^n_bits-1)*xi,k*xj,l
which is a quadratic where every pair of binary variables (xi,k , xj,l) gets a coefficient. λ scales how heavily risk is weighted against return.

**Budget constraint:**
We want sum(wi)=1. Penalty methods enforce equality constraints by squaring (sum(wi)-1)^2 into (sum(wi))^2 -2(sum(wi)) + 1. Constant +1 doesn't affect which x minimizes energy so drop it. Substiture binary encoding of wi into the expansion. For off-diangonal entris col?idx we get cross terms 2wiwj from expansion (i=j and k!=l are different bits of the same asset)(i!=j are entirely different assets). The factor of 2 remains because the QUBO matric is stored in the upper-traingualr Q[idx,col] where idx<col represents the full coefficients and not just half.

**Sector cap constraint:**
Same structure as budget constraint except the binary variables are restricted to sector_cap instead of 1. Its an upstream penalty since the squared penalty  (sum(wi - cap))^2 where i ∈ sector penalizes the deviation both above and below the cap, while we only want sum <= cap so it most only penalize it if sum > cap.

In [2]:
n_assets=5
n_bits=3       #bits per asset weight (gives us 2^3=8)
N=n_assets*n_bits #total binary variables

"""
Class below builds the QUBO matrix Q which has all the portfolio optimization problems.
We want to minimize x^TQ^x where x is the binary vector

What we encode: return maximization, risk minimization, budget constraints, sector cap constraint.

Parameters:
mean_returns : array of expected returns
cov_matrix: covariance matrix
lambda_budget: penalty coefficient for budget constraint
lambda_sector: penalty coefficient for sector cap constraint
sector_map : dictionary mapping asset index to its sector
sector_cap: max budget allocation per sector
"""
def build_qubo_matrix(mean_returns,cov_matrix,lambda_risk,lambda_budget,lambda_sector,sector_map,sector_cap):
    Q = np.zeros((N,N))

    #Returns term: maximize return= minimize negative return
    for i in range(n_assets):
        for k in range(n_bits):
            idx = i * n_bits + k
            weight_contribution = (2**k) /(2**n_bits -1)
            Q[idx,idx] -= mean_returns[i] * weight_contribution

    #Risk term: lambda_risk * w^T * sigma * w converted to binary
    for i in range(n_assets):
        for j in range(n_assets):
            for k in range(n_bits):
                for m in range(n_bits):
                    row = i * n_bits + k
                    col = j * n_bits + m
                    wi = (2**k) /(2**n_bits -1)
                    wj = (2**m)/(2**n_bits -1)
                    Q[row,col] += lambda_risk * cov_matrix[i,j] * wi * wj

    #Budget constraints: (sum_i w_i -1)^2 expanded into binary using binary idempotency so x_i^2=x_i
    for i in range(n_assets):
        for k in range(n_bits):
            idx= i *n_bits + k
            wi = (2**k)/ (2**n_bits-1)
            Q[idx,idx] += lambda_budget * wi * (wi-2) # diagonal terms

            for j in range(n_assets):
                for m in range(n_bits):
                    col = j*n_bits + m
                    if col> idx:
                        wj = (2**m)/(2**n_bits -1)
                        Q[idx,col] += 2 * lambda_budget * wi * wj

    # Sector cap constraint: upstream penalty
    # implemented as a quadratic penalty added to Q
    # for each sector s: (sum_{i in s} w_i - cap_s)^2 if sum > cap_s, else 0
    sectors = set(sector_map.values())
    for sector in sectors:
        sector_assets = [i for i, s in sector_map.items() if s == sector]
        for i in sector_assets:
            for k in range(n_bits):
                idx = i*n_bits +k
                wi = (2**k)/(2**n_bits -1)
                Q[idx,idx] += lambda_sector * wi * (wi -2 * sector_cap)
                for j in sector_assets:
                    for m in range(n_bits):
                        col= j*n_bits + m
                        if col>idx:
                            wj = (2**m)/(2**n_bits-1)
                            Q[idx,col]+= 2 * lambda_sector * wi * wj
    return Q
            

In [3]:
tickers = ["AAPL", "MSFT","GOOGL","JPM","GS"]
data= yf.download(tickers, start = "2020-01-01", end = "2026-01-01")
prices=data["Close"]
log_returns=np.log(prices/prices.shift(1)).dropna()

mean_returns = log_returns.mean().values * 252
cov_matrix = log_returns.cov().values * 252

# scetor map: 0=tech, 1=finance. ticker:sector
sector_map = {0:0,1:0,2:0,3:1,4:1}

Q= build_qubo_matrix(mean_returns=mean_returns,cov_matrix=cov_matrix,lambda_risk=1.0,lambda_budget=5.0
                    ,lambda_sector=2.0,sector_map=sector_map,sector_cap=0.6)
print(f"Q matrix shape: {Q.shape}")


[*********************100%***********************]  5 of 5 completed

Q matrix shape: (15, 15)


**CELL 4**

x^TQx is the QUBO energy function and it bundles the return term, risk term, and penalty terms all in one number. By minimizing it, it pushed towards high return, low risk, whilst obeying the constraints.

Brute force iterates over every possible binary string of N and keeps the lowest energy one. Gives us a benchmark for comparison when we do annealing (and later QAOA)

In [4]:

# Brute force (using small N for tractability)
# N = assets * bits ∴ 2^15 = 32768

def brute_force_qubo(Q): #solves by searching over all 2^N binary strings
    N = Q.shape[0]
    best_energy = np.inf
    best_x = None

    for bits in product([0,1], repeat=N):
        x=np.array(bits)
        energy = x@Q@x
        if energy < best_energy:
            best_energy = energy
            best_x = x.copy
    return best_x, best_energy

x_brute,energy_brute = brute_force_qubo(Q)
print(f"Brute force optimal energy: {energy_brute: .5f}")


Brute force optimal energy: -6.53607


**CELL 5**

SA algorithm simulates the physical annealing of a metal.

num_sweeps mimics temperature cooling while the algorithm runs. 

sum_reads=x allows for x (in this case i did 1000) independent annealing runs in hopes of reaching the local minima (lowest energy). 

reponse.first picks the result with lowest energy across all 1000 runs

**Result from cell 5**

SA optimal energy = Brute force optimal energy ==> SA found the true optimum

In [5]:
# Simulated annealing

#convert Q matrix into dimod formal
Q_dict={}
for i in range(N):
    for j in range(i,N):
        if i == j:
            val = Q[i,j]
        else:
            val = Q[i,j] + Q[j,i] # allows to fold the lower triange in and don't discard it
        if val != 0:
            Q_dict[(i,j)] = val
            
        # if Q[i,j]!=0:
        #     Q_dict[(i,j)] = Q[i,j]

bqm = dimod.BinaryQuadraticModel.from_qubo(Q_dict)
sampler = dimod.SimulatedAnnealingSampler()
response = sampler.sample(bqm,num_reads=1000,num_sweeps=1000)

best_sample = response.first.sample
best_energy= response.first.energy

print(f"Simulated annealing optimal energy: {best_energy:.5f}")

#Return binary solution back to portfolio weights
x_sa = np.array([best_sample[i] for i in range(N)])
weights_qubo = np.array([sum((2**k)* x_sa[i*n_bits +k] for k in range(n_bits)) / (2**n_bits -1) for i in range(n_assets)])
print("\nQUBO Portfolio Weights:")
for ticker, w in zip(tickers,weights_qubo):
    print(f"{ticker}: {w:.3f}")

Simulated annealing optimal energy: -6.53607

QUBO Portfolio Weights:
AAPL: 0.000
MSFT: 0.286
GOOGL: 0.286
JPM: 0.000
GS: 0.429


In [6]:
# Compare to the Markowitz results from the classical portfolio engine

import sys
sys.path.append('/Users/linanachdi/Documents/GitHub/portfolio-optimization-engine')
import portfolio_engine as pe

prices,log_returns = pe.get_data(tickers, "2020-01-01", "2026-01-01")
mean_returns_cont, cov_matrix_cont = pe.annualized_stats(log_returns)
mw_weights,mw_ret,mw_vol,mw_sharpe  = pe.maximum_sharpe(mean_returns_cont,cov_matrix_cont)

print("Markowitz vs QUBO (simulated annealing):")
print(f"{'Asset':<8} {"Markowitz": >12} {"QUBO-SA": >12}")
for ticker, w_mw,w_qubo in zip(tickers,mw_weights,weights_qubo):
    print(f"{ticker:<8} {float(w_mw):>11.2%} {float(w_qubo):>11.2%}")

[*********************100%***********************]  5 of 5 completed

Markowitz vs QUBO (simulated annealing):
Asset       Markowitz      QUBO-SA
AAPL          14.25%       0.00%
MSFT          45.11%      28.57%
GOOGL         40.65%      28.57%
JPM            0.00%       0.00%
GS             0.00%      42.86%


**Analysis of results:**

Notice each percentage in the QUBO SA is a divisor of 7 (0%=0, 28.57%=2/7, 42.86=3/7) which is consistent with the n_bits=3 that I chose, and confirms the discretization logic is correct.

Markowitz found the unconstraineed maximum-Shapre portfolio with no limits and continuous weights. It chose that concentrating 100% in tech (AAPL,GOOGL,MSFT) and 0% in finance (JPM,GS was optimal for risk-adjusted returns given historical data it was given. Meanwhile, the QUBO SA solved a more constrained version of the problem with sector cap enforced. The tech sector cap enforced is 60%, and so both of the sectors in this portfolio remain under that limit (tech=57.14%, finance=42.86). The Markowitz had no such constraint and was able to blow right past that. The QUBO was forced to hold a share in GS because the MArkowitz's optimal solution was sector concentrated and that would the penalty term pushed the solver away from that concentration (cost of diversification). 

The QUBO probably has a lower Sharpe ratio than the MArkowitz portfolio as a tradeoff for better sector diversification.